# 🌀 CycloneX — Worker GPU (Google Colab)

Roda este Colab como **Worker** conectado ao seu **Master** via Ngrok.

## ✅ Antes de começar:
1. Vá em **Runtime → Change runtime type → GPU** (selecione T4 — é grátis)
2. Preencha **apenas a Célula 4** com a URL do Ngrok e o Token
3. Execute todas as células em ordem (`Ctrl+F9` executa tudo)

---
💡 **Dica multi-worker:** Abra este mesmo notebook em várias abas/contas Google diferentes para criar múltiplos workers simultâneos!

## 🔧 Célula 1 — Verificar GPU

In [ ]:
import subprocess, os, sys

print('=== GPU INFO ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('❌ GPU não detectada! Vá em Runtime → Change runtime type → GPU')
print(result.stdout)

# Pega compute capability
cc_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print(f'\n✅ GPU OK: {cc_result.stdout.strip()}')

## 📥 Célula 2 — Clonar repositório CycloneX

In [ ]:
REPO_DIR = '/content/cycloneX'

if os.path.exists(REPO_DIR):
    print('Repositório já existe. Atualizando...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print('Clonando repositório CycloneX...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/ggsofthouse/cycloneX.git', REPO_DIR],
        check=True
    )

os.chdir(REPO_DIR)
print(f'\n✅ Diretório: {os.getcwd()}')
print('Arquivos:', os.listdir('.'))

## ⚙️ Célula 3 — Compilar CUDACyclone para Linux

> O `.exe` do Windows não funciona no Linux. Compilamos o `.cu` com `nvcc` (já instalado no Colab).
>
> ⏱️ Leva **2~4 minutos** na primeira vez.

In [ ]:
CUDA_DIR = f'{REPO_DIR}/CUDACyclone-main'
BINARY   = f'{CUDA_DIR}/CUDACyclone'
SYMLINK  = f'{REPO_DIR}/CUDACyclone.exe'  # Nome que o cyclone_agent.py procura

if os.path.exists(BINARY):
    print(f'✅ Binário já compilado: {BINARY}')
else:
    # Detectar compute capability
    cc_raw = subprocess.run(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True
    ).stdout.strip().replace('.', '')
    GPU_ARCH = cc_raw if cc_raw else '75'
    print(f'GPU Compute Capability: {GPU_ARCH}')
    print(f'\n🔨 Compilando CUDACyclone.cu (aguarde ~3 min)...\n')

    build = subprocess.run(
        ['make', f'GPU_ARCHS={GPU_ARCH}', '-j4'],
        capture_output=True, text=True,
        cwd=CUDA_DIR
    )
    # Mostra últimas linhas do output
    output = (build.stdout + build.stderr)
    print(output[-3000:] if len(output) > 3000 else output)

    if build.returncode != 0 or not os.path.exists(BINARY):
        raise RuntimeError('❌ Build falhou! Veja o erro acima.')
    print(f'\n✅ Build OK!')

# Criar symlink CUDACyclone.exe → binário Linux
# (cyclone_agent.py procura por .exe — symlink resolve sem modificar o código)
if os.path.exists(SYMLINK):
    os.remove(SYMLINK)
os.symlink(BINARY, SYMLINK)
print(f'✅ Symlink criado: CUDACyclone.exe → {BINARY}')

size = os.path.getsize(BINARY) / (1024*1024)
print(f'   Tamanho do binário: {size:.1f} MB')

## 🌐 Célula 4 — ⚠️ CONFIGURAR AQUI (Master URL + Token)

Preencha a URL do Ngrok e o Token do seu Master.

> 🔒 **Segurança:** Estas informações ficam **apenas nesta sessão Colab** — não são salvas em nenhum arquivo nem commitadas no GitHub.

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  👇 EDITE APENAS ESTAS DUAS LINHAS:                 ║
# ╚══════════════════════════════════════════════════════╝

MASTER_URL = "https://XXXX-XX-XX-XX-XX.ngrok-free.app"  # ← URL gerada pelo Ngrok no Master
TOKEN      = "cyclone_token_12345"                       # ← Token do config.yaml do Master

# ── Parâmetros de performance (ajuste por GPU) ──────────
# T4 Colab Free:   GRID="512,1024"  SLICES=256
# A100 Colab Pro:  GRID="1024,1024" SLICES=512
# L4 Colab Pro+:   GRID="512,1024"  SLICES=512
GRID   = "512,1024"
SLICES = 256

# ────────────────────────────────────────────────────────
print(f'Master URL : {MASTER_URL}')
print(f'Token      : {TOKEN[:4]}...{TOKEN[-4:]}  (parcialmente oculto)')
print(f'Grid       : {GRID}')
print(f'Slices     : {SLICES}')

## 📝 Célula 5 — Gerar config.yaml (modo Worker)

In [ ]:
import socket

os.chdir(REPO_DIR)

# Worker ID = hostname único da instância Colab
worker_id = socket.gethostname()
print(f'Worker ID desta instância: {worker_id}')

config_yaml = f"""job:
  grid: \"{GRID}\"
  slices: {SLICES}
  gpus: \"0\"
  random: false

gpu:
  max_temp_c: 85

telemetry:
  interval: 1

api:
  port: 8081
  token: \"{TOKEN}\"

database:
  path: \"./cyclone_worker.db\"

pool:
  role: \"worker\"
  master_url: \"{MASTER_URL}\"
  block_size_gkeys: 200
  random_in_block: false
"""

with open('config.yaml', 'w') as f:
    f.write(config_yaml)

print('\n✅ config.yaml gerado:\n')
print(config_yaml)

## 🔗 Célula 6 — Testar conexão com o Master

In [ ]:
import urllib.request, json, time

print(f'Testando conexão com {MASTER_URL} ...')

for attempt in range(3):
    try:
        req = urllib.request.Request(
            f'{MASTER_URL}/api/status',
            headers={
                'Authorization': f'Bearer {TOKEN}',
                'ngrok-skip-browser-warning': 'true'   # Pula página de aviso do Ngrok
            }
        )
        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.loads(r.read())
        print(f'\n✅ Master acessível!')
        print(f'   Job rodando : {data.get("job_running", "?")}')  
        print(f'   Speed       : {data.get("speed_mkeys", 0):.1f} MKeys/s')
        print(f'   Workers     : {len(data.get("pool_workers", {}))} conectados')
        break
    except Exception as e:
        print(f'Tentativa {attempt+1}/3 falhou: {e}')
        if attempt < 2:
            print('Aguardando 5s...')
            time.sleep(5)
        else:
            print('\n❌ Não foi possível conectar ao Master.')
            print('Verifique:')
            print('  1. O Ngrok está rodando no Master? (ngrok http 8080)')
            print('  2. O cyclone_agent.py está rodando no Master?')
            print('  3. A MASTER_URL está correta?')

## 🚀 Célula 7 — Iniciar Worker CycloneX

> Esta célula roda continuamente até o Colab expirar (~12h grátis).
>
> Acompanhe o progresso no **dashboard do Master**: `http://localhost:8080` → aba **Pool/Workers**

In [ ]:
os.chdir(REPO_DIR)

print('🚀 Iniciando CycloneX Worker...')
print(f'   Master : {MASTER_URL}')
print(f'   Worker : {socket.gethostname()}')
print('   Pressione ■ (Stop) para encerrar\n')
print('=' * 60)

proc = subprocess.Popen(
    [sys.executable, 'cyclone_agent.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)

try:
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    proc.wait()
    print('\n⛔ Worker encerrado.')
finally:
    if proc.poll() is None:
        proc.terminate()

---

## 📊 Informações sobre o Puzzle 71

| Item | Valor |
|------|-------|
| **Endereço alvo** | `1PWo3JeB9jrGwfHDNpdGK54CRas7fsVzXU` |
| **Range de busca** | `400000000000000000` → `7fffffffffffffffff` |
| **Espaço total** | ~4 × 10¹⁷ chaves (enorme!) |
| **T4 Colab** | ~10 GKeys/s |
| **Estratégia** | Pool distribui blocos de 200 GKeys por worker |

## 🏎️ Dica de Performance por GPU do Colab

| GPU | GRID | SLICES | Estimativa |
|-----|------|--------|------------|
| **T4** (grátis) | `512,1024` | `256` | ~8–15 GKeys/s |
| **A100** (Pro) | `1024,1024` | `512` | ~40–60 GKeys/s |
| **L4** (Pro+) | `512,1024` | `512` | ~25–35 GKeys/s |

## 💡 Rodando múltiplos workers

1. Abra este notebook em outra **conta Google**
2. Vá em `File → Save a copy in Drive`
3. Preencha a mesma `MASTER_URL` e `TOKEN`
4. Execute tudo — aparecerá como um novo Worker no dashboard!

---
*CycloneX v2.0 — Distributed GPU Worker*